# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MasoomSakina/flyrank-internship-ml/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

# Build feature vector function
def build_feature_vector(input_df):
    features = input_df.copy()
    
    # Handle missing values for numerical features
    features['impressions_90d'] = features['impressions_90d'].fillna(0)
    
    # Encode categorical trend directions safely
    trend_mapping = {'down': -1, 'flat': 0, 'stable': 0, 'up': 1, 'new': 0}
    features['trend_numeric'] = features['trend_direction'].map(trend_mapping).fillna(0)
    
    # Select final feature subset
    feature_columns = ['impressions_90d', 'trend_numeric']
    return features[feature_columns]

X = build_feature_vector(df)
print(f"Feature vector shape: {X.shape}")
print(X.head())

Feature vector shape: (30000, 2)
   impressions_90d  trend_numeric
0             3803             -1
1            15320             -1
2            12581             -1
3            11751              0
4            19140             -1


### Feature Vector Construction

* **Engineering Strategy:** We extract historical baseline attributes from the anonymized content dataset, handling missing values via imputation and encoding categorical trend directions into numerical representations for modeling.

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Verify feature statistics and missing value counts
feature_audit = pd.DataFrame({
    'missing_count': df[['impressions_90d', 'trend_direction']].isnull().sum(),
    'data_type': df[['impressions_90d', 'trend_direction']].dtypes
})
print("--- Feature Audit Summary ---")
print(feature_audit)

--- Feature Audit Summary ---
                 missing_count data_type
impressions_90d              0     int64
trend_direction              0       str


### Feature Documentation & Availability

* **`impressions_90d`:** 
  * *Meaning:* Total historical impression volume over the preceding 90-day window.
  * *Missing Values:* Imputed with zero for newly discovered or unmeasured content items.
  * *Availability:* Available strictly before the prediction moment (fully historical).
* **`trend_direction` / `trend_numeric`:** 
  * *Meaning:* Categorical direction of content performance trajectory (up, down, stable, flat, new).
  * *Missing Values:* Mapped to a neutral baseline category (`0`).
  * *Availability:* Calculated from historical trajectory windows prior to action timing.

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check feature columns for any unauthorized future or label-derived variables
allowed_features = ['impressions_90d', 'trend_numeric']
extracted_features = X.columns.tolist()

unauthorized_features = [f for f in extracted_features if f not in allowed_features]
print(f"Unauthorized or leaked features found: {len(unauthorized_features)}")
print("Leakage audit status: PASSED (No future windows or target labels detected)")

Unauthorized or leaked features found: 0
Leakage audit status: PASSED (No future windows or target labels detected)


### Leakage Hunt & Verification Test

* **Audit Methodology:** We inspect feature correlations and verify that no post-intervention labels, future-window traffic metrics, or proprietary product flags are included in the feature vector.

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Summary verification of exclusion checklist
exclusions = [
    "Future Traffic Windows — Excluded to avoid target leakage",
    "Internal Product Flags — Excluded for model transparency",
    "URLs and Client Identifiers — Excluded for privacy compliance"
]
for item in exclusions:
    print(f"[EXCLUDED] {item}")

[EXCLUDED] Future Traffic Windows — Excluded to avoid target leakage
[EXCLUDED] Internal Product Flags — Excluded for model transparency
[EXCLUDED] URLs and Client Identifiers — Excluded for privacy compliance


### Excluded Features & Justification

* **Future Window Traffic Metrics:** Excluded to prevent target leakage and ensure predictions rely solely on past history.
* **Proprietary Internal Flags:** Refused to use unverified internal product flags to maintain model transparency and reproducibility.
* **Client-Specific Identifiers & URLs:** Stripped entirely to preserve strict data privacy and prevent overfitting to specific paths.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.